# SessionRerank+ Kaggle v21
Fixes: numpy==1.26.4, transformers==4.44.2, sentence-transformers==2.7.0, hard-fail download, purge numpy+torch cache

In [ ]:
# CELL 1: Pull latest repo
import os, subprocess, sys
REPO_DIR = '/kaggle/working/sr'
if os.path.isdir(REPO_DIR + '/.git'):
    subprocess.run(['git','-C',REPO_DIR,'fetch','origin'], check=True)
    subprocess.run(['git','-C',REPO_DIR,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','https://github.com/joyjeni/session-aware-toolbench-rerank.git',REPO_DIR], check=True)
print('commit:', subprocess.run(['git','-C',REPO_DIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())

In [ ]:
# CELL 2: Install pinned deps - fixes v20 numpy UInt32DType + transformers ModuleNotFoundError
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torch','torchaudio','numpy','transformers','sentence-transformers','accelerate','bitsandbytes'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','setuptools','torch==2.4.1+cu121','--extra-index-url','https://download.pytorch.org/whl/cu121'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','numpy==1.26.4'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2','sentence-transformers==2.7.0','huggingface_hub>=0.23','accelerate==0.30.1','bitsandbytes>=0.43','rank-bm25','scipy>=1.11','scikit-learn>=1.3','pyarrow>=12'], check=True)
print('[install] done')

In [ ]:
# CELL 3: Purge stale numpy+torch from module cache then smoke-test
import importlib, sys
for k in [m for m in list(sys.modules) if m=='numpy' or m.startswith('numpy.') or m=='torch' or m.startswith('torch.')]:
    del sys.modules[k]
importlib.invalidate_caches()
import numpy as np, torch
assert torch.cuda.is_available(), 'No CUDA - enable GPU T4 in Settings'
print('numpy:', np.__version__, '| torch:', torch.__version__, '| CUDA:', torch.cuda.get_device_name(0))

In [ ]:
# CELL 4: Install sessionrerank package
import os, subprocess, sys
os.chdir('/kaggle/working/sr')
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e','.'], check=True)
print('[pkg] done')

In [ ]:
# CELL 5: HF token
import os
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGINGFACE_HUB_TOKEN'] = token
        print('[HF_TOKEN] set')
    else:
        print('[HF_TOKEN] empty - Gemma will be skipped')
except BaseException as e:
    print('[HF_TOKEN] unavailable:', e)

In [ ]:
# CELL 6: Download ToolBench - hard-fail if missing after download (fixes v20 silent failure)
import os, subprocess, sys
from pathlib import Path
os.chdir('/kaggle/working/sr')
DEST = Path('/kaggle/working/sr/data/toolbench_hf')
BENCH = DEST / 'benchmark'
REQUIRED = ['g1_instruction-00000-of-00001.parquet','g1_category-00000-of-00001.parquet','g1_tool-00000-of-00001.parquet','g2_instruction-00000-of-00001.parquet','g2_category-00000-of-00001.parquet','g3_instruction-00000-of-00001.parquet']
if BENCH.exists() and all((BENCH/f).exists() for f in REQUIRED):
    print('[download] already present')
else:
    print('[download] fetching from HuggingFace...')
    subprocess.run([sys.executable,'-m','sessionrerank.data.download','--out',str(DEST)], cwd='/kaggle/working/sr', check=True)
    print('[download] done')
missing = [f for f in REQUIRED if not (BENCH/f).exists()]
if missing:
    raise FileNotFoundError(f'Still missing after download: {missing} - check HF_TOKEN + internet')
print(f'[verify] {len(list(BENCH.glob("*.parquet")))} parquet files at {BENCH}')

In [ ]:
# CELL 7: Verify full import stack
import sys, numpy as np, torch, transformers, sentence_transformers
print('Python:', sys.version.split()[0])
print('numpy:', np.__version__)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence_transformers:', sentence_transformers.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
# CELL 8: Run full evaluation - T4-safe
import subprocess, sys, os, time
os.chdir('/kaggle/working/sr')
cmd = [sys.executable,'-m','sessionrerank.eval.run_all','--device','cuda','--encoder','sentence-transformers/all-MiniLM-L6-v2','--seeds','0,1,2','--fit-sessions','4000','--cooc','--cooc-sessions','20000','--cross-encoder','--cross-encoder-model','cross-encoder/ms-marco-MiniLM-L-6-v2','--cross-encoder-topk','100','--gemma-rerank','google/gemma-2-2b-it','--gemma-rerank-topk','50','--gemma-rerank-batch-size','4','--gemma-4bit','--free-embed-after-encode','--resume','--out','results/metrics.json']
print('[run_all] starting...')
t0 = time.time()
r = subprocess.run(cmd, cwd='/kaggle/working/sr')
print('[run_all] exit', r.returncode, 'elapsed', round(time.time()-t0), 's')

In [ ]:
# CELL 9: Show results
import json, pathlib
p = pathlib.Path('/kaggle/working/sr/results/metrics.json')
if not p.exists():
    pp = p.with_suffix('.json.partial')
    if pp.exists(): p = pp; print('[partial checkpoint]')
if p.exists():
    data = json.loads(p.read_text())
    splits = data.get('splits',{})
    if not splits and 'seeds' in data: splits = (data['seeds'] or [{}])[0].get('splits',{})
    print(f"{'Split':<20} {'Regime':<10} {'Method':<52} {'NDCG@5':>8} {'Hit@5':>8}")
    print('-'*104)
    for split,sb in splits.items():
        for regime,rb in sb.get('regimes',{}).items():
            for method,m in rb.items():
                if isinstance(m,dict) and 'NDCG@5' in m:
                    nd = m['NDCG@5']; hi = m['Hit@5']
                    print(f"{split:<20} {regime:<10} {method:<52} {(nd['mean'] if isinstance(nd,dict) else nd):>8.4f} {(hi['mean'] if isinstance(hi,dict) else hi):>8.4f}")
else: print('[no results found]')